# Notebook 3 — Fine-tuning con LoRA: BERT / clinical-BERT / mT5

Skeleton único y reutilizable para los tres modelos. Para cambiar de modelo, se edita
**solo la celda de configuración** (`MODEL_KEY`) — el resto del notebook se adapta solo.

**Cambios importantes respecto al ejemplo visto en clase:**
- `target_modules` depende de la arquitectura: BERT/RoBERTa usan `query`/`value`. mT5 (arquitectura T5)
  usa `q`/`v`.
- `task_type`: Para BERT/clinical-BERT es `TOKEN_CLS`
  (clasificación de tokens / NER) y mT5 es `SEQ_2_SEQ_LM`.

## 1. Setup e imports

In [1]:
try:
    from google.colab import drive
    drive.mount("/content/drive")

except ImportError:
    SPLITS_DIR = "."

Mounted at /content/drive


In [2]:
# !pip install transformers peft datasets seqeval evaluate accelerate --quiet

import numpy as np
import torch
import random
from pathlib import Path
from datasets import load_from_disk

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForSeq2SeqLM,
    DataCollatorForTokenClassification,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Seq2SeqTrainingArguments,
    Trainer,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model, TaskType

random.seed(42)

SPLITS_DIR = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M1/distemist_final")


## 2. Configuración — ÚNICO LUGAR QUE CAMBIA ENTRE MODELOS

Cambien `MODEL_KEY` a `"bert"`, `"clinical_bert"` o `"mt5"` y vuelvan a correr el notebook
completo. Ajusten los nombres de checkpoint según lo que decidieron usar (recuerden: si el
texto de DisTEMIST está en español, el "BERT clínico" debe ser un modelo clínico en
**español**, no BioClinicalBERT que es en inglés/MIMIC — como vimos antes).


In [3]:
MODEL_KEY = "clinical_bert"  # "bert" | "clinical_bert" | "mt5"

MODEL_REGISTRY = {
    "bert": {
        "checkpoint": "dccuchile/bert-base-spanish-wwm-cased",
        "arch": "token_classification",
        "lora_target_modules": ["query", "value"],
        "lora_task_type": TaskType.TOKEN_CLS,
        "dataset": "distemist_bert_format",
    },
    "clinical_bert": {
        "checkpoint": "PlanTL-GOB-ES/roberta-base-biomedical-clinical-es",
        "arch": "token_classification",
        "lora_target_modules": ["query", "value"],
        "lora_task_type": TaskType.TOKEN_CLS,
        "dataset": "distemist_bert_format",
    },
    "mt5": {
        "checkpoint": "google/mt5-small",
        "arch": "seq2seq",
        "lora_target_modules": ["q", "v"],
        "lora_task_type": TaskType.SEQ_2_SEQ_LM,
        "dataset": "distemist_mt5_format",
    },
}

cfg = MODEL_REGISTRY[MODEL_KEY]
print(cfg)


{'checkpoint': 'PlanTL-GOB-ES/roberta-base-biomedical-clinical-es', 'arch': 'token_classification', 'lora_target_modules': ['query', 'value'], 'lora_task_type': <TaskType.TOKEN_CLS: 'TOKEN_CLS'>, 'dataset': 'distemist_bert_format'}


Login W&B

In [ ]:
! wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: icamachom1 (medical-research-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 3. Cargar datos procesados del Notebook 1

Mismo split para los tres modelos — solo cambia el *formato* según la arquitectura.


In [4]:
if cfg["arch"] == "token_classification":
    raw_ds = load_from_disk(str(SPLITS_DIR / "distemist_bert_format"))
    label_list = ["O", "B-ENFERMEDAD", "I-ENFERMEDAD"]
    label2id = {l: i for i, l in enumerate(label_list)}
    id2label = {i: l for i, l in enumerate(label_list)}
else:
    raw_ds = load_from_disk(str(SPLITS_DIR / "distemist_mt5_format"))

print(raw_ds)


DatasetDict({
    train: Dataset({
        features: ['doc_id', 'tokens', 'ner_tags'],
        num_rows: 48
    })
    dev: Dataset({
        features: ['doc_id', 'tokens', 'ner_tags'],
        num_rows: 5
    })
    test: Dataset({
        features: ['doc_id', 'tokens', 'ner_tags'],
        num_rows: 9
    })
})


In [5]:
print(raw_ds["dev"].column_names)
print("doc_id" in raw_ds["dev"].column_names)  # True/False directo

# y confirma que además tiene contenido real, no solo que la columna existe
print(raw_ds["dev"]["doc_id"][:5])

['doc_id', 'tokens', 'ner_tags']
True
['es-S0210-48062006000700010-1', 'es-S0210-48062009000600018-1_chunk0', 'es-S0210-48062009000600018-1_chunk1', 'es-S1135-76062010000200006-1_chunk0', 'es-S1135-76062010000200006-1_chunk1']


## 4. Tokenizador y modelo base

In [6]:
tokenizer = AutoTokenizer.from_pretrained(cfg["checkpoint"])

if cfg["arch"] == "token_classification":
    model = AutoModelForTokenClassification.from_pretrained(
        cfg["checkpoint"],
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
    )
else:
    model = AutoModelForSeq2SeqLM.from_pretrained(cfg["checkpoint"])

model


config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/540k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  504MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForTokenClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(52000, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (L

## 5. Tokenización / preprocesamiento según arquitectura

In [7]:
def tokenize_and_align_labels_bert(example):
    tokenized = tokenizer(example["tokens"], truncation=True, is_split_into_words=True)
    word_ids = tokenized.word_ids()
    labels = []
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)  # ignorado en la loss
        elif word_idx != previous_word_idx:
            labels.append(label2id[example["ner_tags"][word_idx]])
        else:
            # subpalabra dentro del mismo token: mismo tag pero como I- si era B-
            tag = example["ner_tags"][word_idx]
            tag = tag.replace("B-", "I-")
            labels.append(label2id[tag])
        previous_word_idx = word_idx
    tokenized["labels"] = labels
    return tokenized


def tokenize_mt5(example):
    model_inputs = tokenizer(example["input_text"], truncation=True, max_length=512)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(example["target_text"], truncation=True, max_length=128)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


if cfg["arch"] == "token_classification":
    tokenized_ds = raw_ds.map(tokenize_and_align_labels_bert, batched=False)
    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
else:
    tokenized_ds = raw_ds.map(tokenize_mt5, batched=False)
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

tokenized_ds


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['doc_id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 48
    })
    dev: Dataset({
        features: ['doc_id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 5
    })
    test: Dataset({
        features: ['doc_id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 9
    })
})

In [8]:
print(tokenized_ds["dev"].column_names)
print("doc_id" in tokenized_ds["dev"].column_names)  # True/False directo

# y confirma que además tiene contenido real, no solo que la columna existe
print(tokenized_ds["dev"]["doc_id"][:5])

['doc_id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels']
True
['es-S0210-48062006000700010-1', 'es-S0210-48062009000600018-1_chunk0', 'es-S0210-48062009000600018-1_chunk1', 'es-S1135-76062010000200006-1_chunk0', 'es-S1135-76062010000200006-1_chunk1']


## 6. Métricas comparables entre los 3 modelos

Como discutimos: BERT hace clasificación de tokens y mT5 genera texto libre, así que no se
puede comparar con la misma función "cruda". La solución es reducir la salida de **ambas**
arquitecturas al mismo formato común — **el conjunto de nombres de enfermedad detectados por
documento** — y calcular precisión/recall/F1 sobre ese conjunto. Así el número final es
comparable entre modelos aunque el mecanismo interno sea distinto.


In [ ]:
# --- 1. Helpers de bajo nivel: texto/BIO -> conjunto de entidades por ejemplo ---

def strip_chunk_suffix(doc_id):
    return __import__("re").sub(r"_chunk\d+$", "", doc_id)


def bio_to_entity_set(tokens, tags):
    entities, current = [], []
    for tok, tag in zip(tokens, tags):
        if tag == "B-ENFERMEDAD":
            if current:
                entities.append(" ".join(current))
            current = [tok]
        elif tag == "I-ENFERMEDAD" and current:
            current.append(tok)
        else:
            if current:
                entities.append(" ".join(current))
            current = []
    if current:
        entities.append(" ".join(current))
    return set(e.lower().strip() for e in entities)


def mt5_output_to_entity_set(generated_text):
    if generated_text.strip().lower() == "ninguna":
        return set()
    parts = generated_text.split("[SEP]")
    return set(p.lower().strip() for p in parts if p.strip())


# --- 2. Agregación por documento original (junta los chunks de un mismo caso) ---

def aggregate_entities_by_original_doc(doc_ids, entity_sets):
    grouped = {}
    for doc_id, ents in zip(doc_ids, entity_sets):
        orig_id = strip_chunk_suffix(doc_id)
        grouped.setdefault(orig_id, set()).update(ents)
    return grouped


# --- 3. Métrica: micro-F1 sumando tp/fp/fn por documento (no un set global) ---

def entity_set_prf1(true_entities, pred_entities):
    tp = len(true_entities & pred_entities)
    fp = len(pred_entities - true_entities)
    fn = len(true_entities - pred_entities)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}


def micro_prf1_by_doc(true_by_doc, pred_by_doc):
    """
    Suma tp/fp/fn documento por documento (evita colisiones de nombres de
    entidades entre pacientes distintos) y calcula precision/recall/F1 al final,
    sobre el corpus completo ya reagrupado.
    """
    tp = fp = fn = 0
    all_doc_ids = set(true_by_doc) | set(pred_by_doc)
    for doc_id in all_doc_ids:
        true_ents = true_by_doc.get(doc_id, set())
        pred_ents = pred_by_doc.get(doc_id, set())
        tp += len(true_ents & pred_ents)
        fp += len(pred_ents - true_ents)
        fn += len(true_ents - pred_ents)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}


# --- 4. compute_metrics por arquitectura, ahora con agregación por documento ---

def compute_metrics_bert(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    doc_ids = raw_ds["dev"]["doc_id"]
    tokens_col = raw_ds["dev"]["tokens"]

    pred_sets, true_sets = [], []
    for pred_seq, label_seq, tokens in zip(predictions, labels, tokens_col):
        pred_tags = [id2label[p] for p, l in zip(pred_seq, label_seq) if l != -100]
        true_tags = [id2label[l] for l in label_seq if l != -100]
        # nota: alinear longitudes con 'tokens' requiere el mismo criterio de truncamiento
        # que en tokenize_and_align_labels_bert; para documentos largos revisen max_length.
        pred_sets.append(bio_to_entity_set(tokens[: len(pred_tags)], pred_tags))
        true_sets.append(bio_to_entity_set(tokens[: len(true_tags)], true_tags))

    pred_by_doc = aggregate_entities_by_original_doc(doc_ids, pred_sets)
    true_by_doc = aggregate_entities_by_original_doc(doc_ids, true_sets)

    return micro_prf1_by_doc(true_by_doc, pred_by_doc)


def compute_metrics_mt5(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    doc_ids = raw_ds["dev"]["doc_id"]

    pred_sets = [mt5_output_to_entity_set(t) for t in decoded_preds]
    true_sets = [mt5_output_to_entity_set(t) for t in decoded_labels]

    pred_by_doc = aggregate_entities_by_original_doc(doc_ids, pred_sets)
    true_by_doc = aggregate_entities_by_original_doc(doc_ids, true_sets)

    return micro_prf1_by_doc(true_by_doc, pred_by_doc)


compute_metrics = compute_metrics_bert if cfg["arch"] == "token_classification" else compute_metrics_mt5

## 7. Baseline (antes de fine-tuning)

Evalúan el modelo base (sin LoRA) para tener el número de referencia con el que van a comparar
el modelo fine-tuneado — necesario para poder afirmar "el fine-tuning mejoró X puntos de F1"
con respaldo real, no solo asumido.


In [ ]:
config = {
    "model": cfg["checkpoint"],
    "arch": cfg["arch"],
    "dataset": cfg["dataset"],
}

In [ ]:
import wandb

wandb.init(
    # Team name
    entity="medical-research-ai",
    # Project
    project="M1-fine-tuning",
    # Name of the run (nombre del modelo )
    name="baseline-roberta",
    # Track hyperparameters and run metadata.
    config=config
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: icamachom1 (medical-research-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
baseline_args_kwargs = dict(
    output_dir="./baseline-eval",
    per_device_eval_batch_size=4,
    report_to="wandb",
)

if cfg["arch"] == "token_classification":
    baseline_args = TrainingArguments(**baseline_args_kwargs)
    baseline_trainer = Trainer(
        model=model,
        args=baseline_args,
        eval_dataset=tokenized_ds["dev"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
else:
    baseline_args = Seq2SeqTrainingArguments(**baseline_args_kwargs, predict_with_generate=True)
    baseline_trainer = Seq2SeqTrainer(
        model=model,
        args=baseline_args,
        eval_dataset=tokenized_ds["dev"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

baseline_metrics = baseline_trainer.evaluate()
print("Métricas BASELINE (sin fine-tuning):", baseline_metrics)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Precision,Recall,F1
No log,1.014463,0,0.003846,0.166667,0.007519


Métricas BASELINE (sin fine-tuning): {'eval_loss': 1.0144630670547485, 'eval_precision': 0.0038461538461538464, 'eval_recall': 0.16666666666666666, 'eval_f1': 0.007518796992481203}


In [ ]:
wandb.finish()

eval/f1,▁
eval/loss,▁
eval/model_preparation_time,▁
eval/precision,▁
eval/recall,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
train/epoch,▁
train/global_step,▁
eval/f1,0.00752


## 8. Configuración de LoRA (corregida por arquitectura)

In [ ]:
config = {
    "model": cfg["checkpoint"],
    "arch": cfg["arch"],
    "dataset": cfg["dataset"],
}

In [ ]:
wandb.init(
    # Team name
    entity="medical-research-ai",
    # Project
    project="M1-fine-tuning",
    # Name of the run (nombre del modelo )
    name="fine-tuned-roberta",
    # Track hyperparameters and run metadata.
    config=config
)

In [ ]:
lora_config = LoraConfig(
    r=8,                                    # rank de las matrices nuevas
    lora_alpha=16,                          # regla común: alpha = 2 x rank
    target_modules=cfg["lora_target_modules"],
    lora_dropout=0.05,
    task_type=cfg["lora_task_type"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

## 9. Argumentos de entrenamiento

In [ ]:
train_args_kwargs = dict(
    output_dir=f"./lora-out-{MODEL_KEY}",
    learning_rate=2e-4,
    num_train_epochs=10,          # con pocos ejemplos (DisTEMIST es un corpus chico), varias épocas ayudan
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=5,
    report_to="none",             # cambien a "wandb" si configuraron Weights & Biases
    fp16=torch.cuda.is_available(),
)

if cfg["arch"] == "token_classification":
    train_args = TrainingArguments(**train_args_kwargs)
    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["dev"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
else:
    train_args = Seq2SeqTrainingArguments(**train_args_kwargs, predict_with_generate=True)
    trainer = Seq2SeqTrainer(
        model=model,
        args=train_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["dev"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

trainer.train()


## 10. Evaluación final en test

In [ ]:
final_metrics = trainer.evaluate(tokenized_ds["test"])
print(f"Métricas FINE-TUNEADO ({MODEL_KEY}) en test:", final_metrics)

print("\n--- Comparación ---")
print("Baseline (dev):     ", baseline_metrics)
print("Fine-tuneado (test):", final_metrics)


## 11. Visualizar un ejemplo: antes vs. después del fine-tuning

Toman un ejemplo del set de test y comparan la predicción del modelo base contra la del
modelo fine-tuneado con LoRA, lado a lado.


In [ ]:
example = raw_ds["test"][0]
print("Texto/tokens de entrada:")
print(example)

# Para reproducir la predicción "antes", necesitan mantener una copia del modelo base
# sin el adaptador LoRA cargado encima (o recargarlo desde cfg["checkpoint"]).
# Ejemplo para mT5 (generativo, más directo de inspeccionar):
if cfg["arch"] == "seq2seq":
    inputs = tokenizer(example["input_text"], return_tensors="pt")
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=64)
    print("\nPredicción del modelo (con LoRA aplicado):")
    print(tokenizer.decode(output_ids[0], skip_special_tokens=True))
    print("\nReferencia (ground truth):")
    print(example["target_text"])
else:
    inputs = tokenizer(example["tokens"], is_split_into_words=True, return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred_ids = logits.argmax(dim=-1)[0].tolist()
    pred_tags = [id2label[i] for i in pred_ids]
    print("\nPredicción del modelo (con LoRA aplicado, a nivel de subtoken):")
    print(pred_tags)
    print("\nReferencia (ground truth, a nivel de token):")
    print(example["ner_tags"])


## 12. Guardar modelo

Con LoRA solo se guardan los pesos del adaptador (mucho más liviano que el modelo completo).


In [ ]:
save_dir = f"./saved-models/{MODEL_KEY}-distemist-lora"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Modelo guardado en: {save_dir}")
